# Lab 08-01: Evaluate an Agent Against a Golden Dataset

**Book chapter:** Chapter 08: Evaluating Agents  
**Estimated time:** 25 minutes

## Overview

Cymbal Retail's platform team wants every change to the order assistant to be checked
automatically before release. In this lab, you build a golden dataset as an ADK eval
set, set thresholds in `test_config.json`, run the evaluation with `AgentEvaluator`, and
confirm that it catches a regression.

## Learning Objectives

By the end of this lab, you will be able to:

- Create an ADK eval set with expected tool calls and reference answers
- Set evaluation criteria and thresholds in test_config.json
- Run AgentEvaluator and interpret its results
- Detect a trajectory regression before release

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. No resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Pillar | Question | Criterion in this lab |
|---|---|---|
| **Process and trajectory** | Did the agent take the right steps? | `tool_trajectory_avg_score` |
| **Success and quality** | Is the final answer right? | `response_match_score` |

## Setup

In [ ]:
%pip install --quiet --upgrade "google-adk[eval]"

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Define the agent under test

The agent under test looks up orders with `get_order_status`. `build_agent` creates the
agent from an instruction, so you can evaluate the original agent and a changed one
with the same eval set.

In [ ]:
import asyncio
import json
import os
import pathlib
from concurrent.futures import ThreadPoolExecutor

from google.adk.agents import Agent
from google.adk.evaluation.agent_evaluator import AgentEvaluator
from google.adk.evaluation.eval_case import EvalCase, IntermediateData
from google.adk.evaluation.eval_case import Invocation
from google.adk.evaluation.eval_set import EvalSet
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")

ORDERS = {"10421": {"item": "Trailhead boots", "status": "shipped",
                    "arrives": "Friday"},
          "10422": {"item": "Rain jacket", "status": "delivered"}}


def get_order_status(order_id: str) -> dict:
    """Returns the item and status of a Cymbal Retail order."""
    order = ORDERS.get(order_id)
    return {"found": True, **order} if order else {"found": False}


def build_agent(instruction):
    return Agent(name="order_assistant", model=MODEL,
                 instruction=instruction, tools=[get_order_status])


GOOD = ("You help shoppers track orders. Always call get_order_status "
        "with the order number before answering, and never guess. If the "
        "order is not found, say so and ask the shopper to check the "
        "number. Answer in one sentence.")

## Step 2: Write the golden dataset as an ADK eval set

A **golden dataset** pairs each question with the expected behavior. Each `EvalCase`
records the user's question, the **expected tool call**, and a **reference answer**.
Building the cases from ADK's own types guarantees a valid `.evalset.json` file.

In [ ]:
def case(case_id, question, order_id, reference):
    """Creates one eval case with its expected tool call and answer."""
    return EvalCase(eval_id=case_id, conversation=[Invocation(
        invocation_id=f"{case_id}-1",
        user_content=types.Content(role="user",
                                   parts=[types.Part(text=question)]),
        final_response=types.Content(role="model",
                                     parts=[types.Part(text=reference)]),
        intermediate_data=IntermediateData(tool_uses=[types.FunctionCall(
            name="get_order_status", args={"order_id": order_id})]))])


golden = EvalSet(eval_set_id="order_golden", eval_cases=[
    case("shipped_order", "Where is order 10421?", "10421",
         "Order 10421, the Trailhead boots, has shipped and should arrive "
         "on Friday."),
    case("delivered_order", "Has my rain jacket from order 10422 arrived?",
         "10422", "Yes, order 10422, the rain jacket, has been delivered."),
    case("unknown_order", "What is the status of order 99999?", "99999",
         "I couldn't find order 99999. Please check the order number."),
])
FOLDER = pathlib.Path("order_eval")
FOLDER.mkdir(exist_ok=True)
EVALSET = FOLDER / "order_golden.evalset.json"
EVALSET.write_text(golden.model_dump_json(indent=2, exclude_none=True))
print(f"Wrote {EVALSET} with {len(golden.eval_cases)} cases")

**Expected result:** `Wrote order_eval/order_golden.evalset.json with 3 cases`.

## Step 3: Set the pass thresholds in test_config.json

`test_config.json`, in the same folder as the eval set, sets the pass thresholds:

| Criterion | Measures | Threshold |
|---|---|---|
| `tool_trajectory_avg_score` | Whether the expected tool calls were made | 1.0 (exact) |
| `response_match_score` | ROUGE-1 word overlap with the reference | 0.4 |

In [ ]:
CRITERIA = {"tool_trajectory_avg_score": 1.0,  # Exact expected tool calls
            "response_match_score": 0.4}  # ROUGE-1 against the reference
(FOLDER / "test_config.json").write_text(
    json.dumps({"criteria": CRITERIA}, indent=2))
print("Thresholds:", CRITERIA)

## Step 4: Run the evaluation

`AgentEvaluator.evaluate` runs every case against the agent, prints a detailed table for
each criterion, and raises an `AssertionError` if any threshold is missed. The same call
runs in pytest, which is how evaluation becomes a CI/CD quality gate.

In [ ]:
def evaluate(agent):
    """Runs the eval set against an agent and reports pass or fail."""
    run = AgentEvaluator.evaluate(
        agent_module=agent, eval_dataset_file_path_or_dir=str(EVALSET),
        num_runs=1)
    try:
        with ThreadPoolExecutor(1) as pool:  # Works in notebooks too.
            pool.submit(asyncio.run, run).result()
        print("\nRESULT: every case met the thresholds")
    except AssertionError as failure:
        print(f"\nRESULT: the evaluation failed\n{failure}")


evaluate(build_agent(GOOD))

**Expected result:** detailed tables, then `RESULT: every case met the thresholds`.

## Step 5: Introduce a regression and run the same evaluation again

The regressed instruction tells the agent to answer from memory, which is the kind of change that looks harmless in a code review.

In [ ]:
REGRESSED = ("You help shoppers track orders. Answer quickly from what "
             "you already know, and call tools only if you are unsure.")
evaluate(build_agent(REGRESSED))

**Expected result:** `RESULT: the evaluation failed`, with
`tool_trajectory_avg_score` at 0.0 for each case, because the agent stopped calling
the tool.

## Step 6: Experiment

- In Step 3, raise `response_match_score` to 0.8 and rerun Steps 3 and 4. A strict
  word-overlap threshold can fail correct answers that are worded differently.
- In Step 2, add a fourth case of your own and rerun Steps 2 to 4.

## Check Your Understanding

**1. Why did the regressed agent fail even though its answers sounded plausible?**

<details><summary>Show answer</summary>

It stopped calling get_order_status, so the trajectory no longer matched the expected tool calls. Evaluating only the final answer would have missed this.

</details>

**2. Why is the ROUGE-1 threshold set low?**

<details><summary>Show answer</summary>

ROUGE-1 measures word overlap, so correct answers with different wording can score low. Semantic or rubric-based judges are better for meaning.

</details>

**3. How does this lab become a CI/CD quality gate?**

<details><summary>Show answer</summary>

The same AgentEvaluator call runs in pytest in the build pipeline, and a failed threshold fails the build.

</details>

## Summary

You built a golden dataset, evaluated an agent's trajectory and responses against it, and
caught a regression that a quick review could have missed. Delete the `order_eval` folder
when you no longer need it.